In [2]:
import json
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest


# ---------- Settings ----------
INPUT_PATH = Path("train_cleaned.parquet")
OUTPUT_PATH = Path("account_features.csv")

DUST_XRP = 0.001
QUICK_FORWARD_MIN = 10


# ---------- Load the cleaned dataset ----------
df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df):,} transactions")

# Destination and DestinationTag are inside this JSON column.
def parse_extra(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            result = json.loads(value)
            return result if isinstance(result, dict) else {}
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


extra = df["EXTRA_TRANSACTION_FIELDS"].map(parse_extra)

# Restrict this graph to straightforward native-XRP payments.
# Other transaction types/assets need separate handling.
eligible = (
    df["TRANSACTION_TYPE"].eq("Payment")
    & df["SUCCESS"].eq(True)
    & df["is_xrp_payment"].eq(1)
    & df["delivered_currency"].eq("XRP")
    & df["transfer_count"].eq(1)
    & df["has_paths"].eq(0)
    & df["has_sendmax"].eq(0)
)

payments = df.loc[eligible, ["TRANSACTION_HASH", "ACCOUNT"]].copy()
payments = payments.rename(columns={"ACCOUNT": "src"})

payments["dst"] = extra.loc[eligible].map(
    lambda fields: fields.get("Destination")
)
payments["tag"] = extra.loc[eligible].map(
    lambda fields: fields.get("DestinationTag")
)
payments["time"] = pd.to_datetime(
    df.loc[eligible, "DATETIME"], utc=True, errors="coerce"
)

# delivered_value is already XRP: do not divide by 1,000,000.
payments["xrp"] = pd.to_numeric(
    df.loc[eligible, "delivered_value"], errors="coerce"
)

valid = (
    payments["src"].map(lambda x: isinstance(x, str) and bool(x))
    & payments["dst"].map(lambda x: isinstance(x, str) and bool(x))
    & payments["time"].notna()
    & payments["xrp"].notna()
    & np.isfinite(payments["xrp"])
    & payments["xrp"].ge(0)
)

print(f"{eligible.sum():,} eligible native-XRP payments")
print(f"{(~valid).sum():,} eligible rows excluded for invalid/missing fields")

payments = payments.loc[valid].copy()

if payments["TRANSACTION_HASH"].duplicated().any():
    raise ValueError("Duplicate transaction hashes: resolve before aggregation.")


# ---------- Step 1: separate dust ----------
# This is an analytical threshold, not proof that a payment is spam.
dust = payments.loc[payments["xrp"] < DUST_XRP].copy()
value = payments.loc[payments["xrp"] >= DUST_XRP].copy()

print(f"{len(dust):,} dust payments set aside")
print(f"{len(value):,} value payments kept")

if value.empty:
    raise ValueError("No value payments remain after filtering.")


# ---------- Step 2: account behaviour ----------
out = value.groupby("src").agg(
    out_n=("xrp", "size"),
    out_xrp=("xrp", "sum"),
    out_cpty=("dst", "nunique"),
    first_out=("time", "min"),
    out_size_cv=(
        "xrp",
        lambda s: s.std() / s.mean() if len(s) > 1 else np.nan,
    ),
)

inc = value.groupby("dst").agg(
    in_n=("xrp", "size"),
    in_xrp=("xrp", "sum"),
    in_cpty=("src", "nunique"),
    first_in=("time", "min"),
    # Missing destination tags are excluded from nunique.
    in_tags=("tag", "nunique"),
)

feat = out.join(inc, how="outer")
feat.index.name = "ACCOUNT"

count_columns = [
    "out_n", "out_xrp", "out_cpty",
    "in_n", "in_xrp", "in_cpty", "in_tags",
]
feat[count_columns] = feat[count_columns].fillna(0)

# Similar incoming/outgoing totals within this dataset.
# This does not trace individual funds or establish forwarding.
both = feat["in_xrp"].gt(0) & feat["out_xrp"].gt(0)

feat["pass_through"] = 0.0
feat.loc[both, "pass_through"] = (
    feat.loc[both, ["in_xrp", "out_xrp"]].min(axis=1)
    / feat.loc[both, ["in_xrp", "out_xrp"]].max(axis=1)
)

# Preserve the original first-event heuristic with a clearer name.
# Negative values mean the first observed send preceded the first receipt.
feat["minutes_first_in_to_first_out"] = (
    feat["first_out"] - feat["first_in"]
).dt.total_seconds() / 60

feat["fan_out"] = feat["out_cpty"]
feat["fan_in"] = feat["in_cpty"]

feat["uniform_splits"] = (
    feat["out_n"].ge(5)
    & feat["out_size_cv"].fillna(1).lt(0.05)
)


# ---------- Step 3: transparent screening flags ----------
feat["flag_rapid_first_event"] = (
    feat["pass_through"].ge(0.9)
    & feat["minutes_first_in_to_first_out"].between(
        0, QUICK_FORWARD_MIN
    )
)
feat["flag_fan_out"] = feat["fan_out"].ge(10)
feat["flag_fan_in"] = feat["fan_in"].ge(10)
feat["flag_uniform_splits"] = feat["uniform_splits"]

flags = [
    "flag_rapid_first_event",
    "flag_fan_out",
    "flag_fan_in",
    "flag_uniform_splits",
]
feat["rule_score"] = feat[flags].sum(axis=1)


# ---------- Step 4: NetworkX cycles ----------
G = nx.DiGraph()

# Multiple payments between the same pair become one weighted edge.
edges = value.groupby(["src", "dst"], as_index=False).agg(
    xrp=("xrp", "sum")
)

G.add_weighted_edges_from(
    edges[["src", "dst", "xrp"]].itertuples(index=False, name=None),
    weight="xrp",
)

# Stream cycles instead of storing every cycle in a list.
# Enumeration can still be expensive in large/dense graphs.
in_cycle = set()
cycle_count = 0

for cycle in nx.simple_cycles(G, length_bound=4):
    if len(cycle) > 1:
        cycle_count += 1
        in_cycle.update(cycle)

feat["in_short_cycle"] = feat.index.isin(in_cycle)

print(f"{cycle_count:,} directed cycles of 2–4 hops found")


# ---------- Step 5: Isolation Forest ----------
# Do not include is_suspicious: it is a label, not an input feature.
model_columns = [
    "out_n", "out_xrp", "out_cpty",
    "in_n", "in_xrp", "in_cpty",
    "pass_through",
]

X = np.log1p(feat[model_columns].clip(lower=0))

model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)
model.fit(X)

# Higher score means more unusual account behaviour.
feat["anomaly_score"] = -model.score_samples(X)


# ---------- Rank and save ----------
# Rules take priority; anomaly score breaks ties.
ranked = feat.sort_values(
    ["rule_score", "anomaly_score"],
    ascending=[False, False],
)

ranked.to_csv(OUTPUT_PATH, index=True)

print(f"\nSaved {len(ranked):,} account rows to {OUTPUT_PATH}")

print("\nFlag counts:")
print(feat[flags + ["in_short_cycle"]].sum().to_string())

review_columns = [
    "rule_score",
    "anomaly_score",
    "in_xrp",
    "out_xrp",
    "fan_in",
    "fan_out",
    "pass_through",
    "minutes_first_in_to_first_out",
    "in_short_cycle",
]

print("\nTop 10 accounts to review:")
print(ranked[review_columns].head(10).round(3).to_string())

FileNotFoundError: [Errno 2] No such file or directory: 'train_cleaned.parquet'